# 08 - Probabilistic Entity Resolution

## Objective

Menilai kesiapan data dan candidate pair untuk probabilistic entity resolution. Notebook ini membangun comparison vector dan evidence table, tetapi tidak mengklaim probabilitas, precision, recall, atau F1 tanpa ground truth.

## Problem

Candidate blocking sudah tersedia, tetapi agreement antar-field belum diterjemahkan menjadi fitur yang dapat dipakai oleh model probabilistic.

## Hypothesis

- Candidate pair dengan agreement pada beberapa field identitas memiliki evidence yang lebih kuat daripada pair yang hanya berbagi satu field.
- Candidate pair hasil blocking yang berbeda dapat digabung dan dianalisis tanpa membuat duplicate pair.
- Tanpa label pasangan dan library probabilistic yang tervalidasi, tahap ini seharusnya berhenti pada feature preparation dan readiness assessment.

## Decision boundary

Notebook ini tidak melakukan automatic merge, clustering final, atau estimasi probabilitas. `customer_id` tidak digunakan sebagai label. Persetujuan bahwa dua record adalah entity yang sama memerlukan ground truth atau manual review.

## Current limitation

Library seperti `recordlinkage`, `Splink`, dan `dedupe` harus diperiksa di environment sebelum digunakan. Jika belum tersedia, notebook hanya menyiapkan input yang reproducible untuk eksperimen berikutnya.

In [ ]:
from pathlib import Path
from importlib.util import find_spec

import pandas as pd

DATA_CANDIDATES = [
    Path.cwd() / 'data' / 'processed' / 'crm_50000_customers_standardized.csv',
    Path.cwd().parent / 'data' / 'processed' / 'crm_50000_customers_standardized.csv',
]
DATA_PATH = next((path.resolve() for path in DATA_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Dataset terstandardisasi tidak ditemukan. Jalankan 04_standardization.ipynb terlebih dahulu.')

PAIR_PATH = DATA_PATH.parent / 'blocking_candidate_pairs.csv'
if not PAIR_PATH.exists():
    raise FileNotFoundError('Artifact candidate pairs tidak ditemukan. Jalankan 07_blocking_candidate_generation.ipynb terlebih dahulu.')

df = pd.read_csv(DATA_PATH).reset_index(names='row_index')
blocking_pairs = pd.read_csv(PAIR_PATH)

required_columns = [
    'row_index', 'email_std', 'phone_digits_std', 'name_key_std',
    'address_std', 'city_std', 'dob_std',
]
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f'Kolom input belum tersedia: {missing_columns}')

print('Dataset:', DATA_PATH)
print('Dataset shape:', df.shape)
print('Blocking pair rows:', blocking_pairs.shape)

## Experiment 1 - Probabilistic method readiness and dependency audit

Ketersediaan library bukan bukti bahwa metode sudah sesuai. Audit ini hanya mendokumentasikan apakah dependency tersedia dan apakah ground truth pasangan tersedia di input saat ini.

In [ ]:
probabilistic_libraries = ['recordlinkage', 'splink', 'dedupe']
dependency_summary = pd.DataFrame([
    {
        'library': library,
        'installed': find_spec(library) is not None,
    }
    for library in probabilistic_libraries
])

ground_truth_columns = [
    column for column in df.columns
    if any(keyword in column.lower() for keyword in ['master', 'entity', 'duplicate_label', 'match_label'])
]
readiness_summary = pd.DataFrame({
    'check': [
        'probabilistic_library_available',
        'ground_truth_label_column_detected',
        'blocking_pair_artifact_available',
    ],
    'result': [
        bool(dependency_summary['installed'].any()),
        bool(ground_truth_columns),
        PAIR_PATH.exists(),
    ],
    'detail': [
        ', '.join(dependency_summary.loc[dependency_summary['installed'], 'library']) or 'none',
        ', '.join(ground_truth_columns) or 'none detected',
        str(PAIR_PATH),
    ],
})

dependency_summary, readiness_summary

## Experiment 2 - Deduplicate candidate pair artifact

Satu pair dapat muncul dari beberapa blocking strategy. Pair disatukan berdasarkan row index dan provenance strategy dipertahankan. Ini mencegah pair yang sama dihitung berkali-kali pada comparison vector.

In [ ]:
required_pair_columns = [
    'blocking_strategy', 'left_row_index', 'right_row_index', 'block_size',
]
missing_pair_columns = sorted(set(required_pair_columns) - set(blocking_pairs.columns))
if missing_pair_columns:
    raise ValueError(f'Kolom artifact pair belum tersedia: {missing_pair_columns}')

blocking_pairs['pair_key'] = list(zip(
    blocking_pairs['left_row_index'].astype(int),
    blocking_pairs['right_row_index'].astype(int),
))

pair_provenance = (
    blocking_pairs.groupby('pair_key', sort=True)
    .agg(
        supporting_blocking_strategies=('blocking_strategy', lambda values: '|'.join(sorted(set(values)))),
        blocking_strategy_count=('blocking_strategy', 'nunique'),
        max_block_size=('block_size', 'max'),
    )
    .reset_index()
)
pair_provenance[['left_row_index', 'right_row_index']] = pd.DataFrame(
    pair_provenance['pair_key'].tolist(), index=pair_provenance.index
)
pair_provenance = pair_provenance.drop(columns='pair_key')

print('Unique candidate pairs:', len(pair_provenance))
pair_provenance.head()

## Experiment 3 - Build comparison vectors

Setiap agreement indicator bernilai 1 jika kedua nilai tersedia dan sama persis setelah standardisasi. Missing value tidak dianggap agreement. Fitur ini adalah input model, bukan keputusan match.

In [ ]:
comparison_fields = {
    'email': 'email_std',
    'phone': 'phone_digits_std',
    'name': 'name_key_std',
    'address': 'address_std',
    'city': 'city_std',
    'dob': 'dob_std',
}

left_values = df.set_index('row_index')[[*comparison_fields.values()]].add_suffix('_left')
right_values = df.set_index('row_index')[[*comparison_fields.values()]].add_suffix('_right')
comparison = pair_provenance.copy()
comparison = comparison.join(left_values, on='left_row_index')
comparison = comparison.join(right_values, on='right_row_index')

agreement_columns = []
for label, column in comparison_fields.items():
    left_column = f'{column}_left'
    right_column = f'{column}_right'
    agreement_column = f'{label}_agree'
    available = comparison[left_column].notna() & comparison[right_column].notna()
    comparison[agreement_column] = (available & comparison[left_column].eq(comparison[right_column])).astype('int8')
    agreement_columns.append(agreement_column)

comparison['agreement_count'] = comparison[agreement_columns].sum(axis=1)
comparison['available_field_count'] = sum(
    (comparison[f'{column}_left'].notna() & comparison[f'{column}_right'].notna()).astype('int8')
    for column in comparison_fields.values()
)
comparison['agreement_pattern'] = comparison[agreement_columns].astype(str).agg(''.join, axis=1)

comparison_summary = (
    comparison.groupby('agreement_count')
    .size()
    .rename('pair_count')
    .reset_index()
    .sort_values('agreement_count')
)
comparison_summary

## Experiment 4 - Candidate evidence review

Ringkasan ini membantu memilih sample manual review dan memisahkan pair dengan evidence rendah, sedang, dan tinggi. Band bukan probabilitas dan tidak dipakai untuk merge otomatis.

In [ ]:
def evidence_band(agreement_count: int) -> str:
    if agreement_count >= 4:
        return 'high_agreement_review'
    if agreement_count >= 2:
        return 'medium_agreement_review'
    return 'low_agreement_review'

comparison['evidence_band'] = comparison['agreement_count'].map(evidence_band)
evidence_summary = (
    comparison.groupby('evidence_band')
    .agg(
        pair_count=('agreement_count', 'size'),
        mean_agreement_count=('agreement_count', 'mean'),
        max_block_size=('max_block_size', 'max'),
    )
    .reset_index()
)
evidence_summary

## Experiment 5 - Persist probabilistic input artifact

Artifact hanya berisi row index, provenance blocking, agreement indicators, dan ringkasan evidence. Nilai customer tidak disimpan ulang. Kolom raw dan `customer_id` tidak digunakan sebagai model feature.

In [ ]:
OUTPUT_DIR = DATA_PATH.parent
COMPARISON_OUTPUT_PATH = OUTPUT_DIR / 'probabilistic_comparison_vectors.csv'
READINESS_OUTPUT_PATH = OUTPUT_DIR / 'probabilistic_readiness_summary.csv'

artifact_columns = [
    'left_row_index', 'right_row_index',
    'supporting_blocking_strategies', 'blocking_strategy_count', 'max_block_size',
    *agreement_columns, 'agreement_count', 'available_field_count',
    'agreement_pattern', 'evidence_band',
]
comparison[artifact_columns].to_csv(COMPARISON_OUTPUT_PATH, index=False)
readiness_summary.to_csv(READINESS_OUTPUT_PATH, index=False)

print('Saved:', COMPARISON_OUTPUT_PATH)
print('Saved:', READINESS_OUTPUT_PATH)
print('Raw dataset still exists:', (DATA_PATH.parents[1] / 'raw' / 'crm_50000_customers_dirty_v3.csv').exists())

# Result, Analysis, and Decision

## Result aktual

Gunakan `dependency_summary`, `readiness_summary`, `comparison_summary`, dan `evidence_summary` sebagai sumber hasil aktual. Angka tidak ditulis manual karena bergantung pada artifact blocking dan snapshot dataset.

## Analysis

- Comparison vector menyatakan agreement per field, bukan label entity.
- Agreement tinggi adalah kandidat untuk manual review, bukan bukti ground truth.
- Agreement rendah dapat menunjukkan noise, missingness, atau blocking yang terlalu longgar.
- Pair yang muncul dari beberapa blocking strategy memiliki provenance lebih kaya, tetapi tetap tidak otomatis benar.

## Decision rule

Probabilistic model hanya boleh dijalankan setelah tersedia minimal satu dari dua hal berikut: ground truth pasangan yang direview, atau desain unsupervised yang asumsi m/u dan validasinya didokumentasikan. Tanpa itu, notebook berhenti pada feature preparation dan readiness assessment.

## Limitations

- Tidak ada ground truth entity pada snapshot saat ini.
- `customer_id` sengaja tidak dipakai sebagai label.
- Belum ada estimasi probabilitas terkalibrasi, precision, recall, atau F1.
- Artifact pair berasal dari blocking yang sudah dipilih pada tahap 07, sehingga pair di luar candidate set belum dinilai.

## Next Experiment

Lakukan manual review terstratifikasi berdasarkan `evidence_band` dan `supporting_blocking_strategies`, lalu simpan label review terpisah. Setelah label tersedia, barulah uji `recordlinkage`, Splink, atau metode probabilistic lain dengan evaluasi yang dapat dipertanggungjawabkan.